# Example 0.13.2 / 4: Exercise 7-1: Restricting Multicommodity Data

Companion notebook for **0.13.2 Exercises on Sets, Parameters and Variables** in the AMPL lab Documentation, section "Exercise 7-1: Restricting Multicommodity Data". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Exercise 7-1: Restricting Multicommodity Data

Every check passes, so AMPL generates and solves the model. When the data break a condition, the
failing `check` stops the solve before the solver runs. With the demand for coils at LAN lowered to
600, the balance check for each product fails. The model below keeps only that check:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ORIG;                                # origins
set DEST;                                # destinations
set PROD;                                # products
param supply {ORIG, PROD} >= 0;          # tons available
param demand {DEST, PROD} >= 0;          # tons required

check {p in PROD}: sum {i in ORIG} supply[i,p] = sum {j in DEST} demand[j,p];  # balance per product

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ORIG"] = ["GARY", "CLEV"]
ampl.set["DEST"] = ["FRA", "DET", "LAN"]
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["supply"] = {("GARY", "bands"): 300, ("GARY", "coils"): 700,
                        ("CLEV", "bands"): 600, ("CLEV", "coils"): 1200}
ampl.param["demand"] = {("FRA", "bands"): 300, ("FRA", "coils"): 650,
                        ("DET", "bands"): 300, ("DET", "coils"): 600,
                        ("LAN", "bands"): 300, ("LAN", "coils"): 600}

In [5]:
# run.py
ampl.eval("check;")

check['coils'] at line 7 of model.mod fails:
  check{p in PROD} : sum{i in ORIG} supply[i,p] == sum{j in DEST} demand[j,p];
